## import thư viện 

In [128]:
import pandas as pd
import re
import numpy as np
from datetime import datetime, timedelta
import random
import unicodedata

## Xử lý các file về chung cấu trúc, gộp các file lại với nhau

### xử lý file excel

In [129]:
# Đọc file và gán loại BĐS
files = {
    'nd-hcm.xlsx': 'Nhà Đất Thổ Cư',
    'nd-hn.xlsx': 'Nhà Đất Thổ Cư',
    'nd-bd.xlsx': 'Nhà Đất Thổ Cư',
    'nd-dn.xlsx': 'Nhà Đất Thổ Cư',
    'cc-hcm.xlsx': 'Căn Hộ Chung Cư',
    'cc-hn.xlsx': 'Căn Hộ Chung Cư'
}

dfs = []

for file, loai_bds in files.items():
    dfr1 = pd.read_excel(file)
    dfr1['Loại BĐS'] = loai_bds
    dfs.append(dfr1)

# Gộp tất cả thành một DataFrame
df1 = pd.concat(dfs, ignore_index=True)
df1

,Tên dự án,Giá,Diện tích,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,"Quận 9, Hồ Chí Minh",4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,"Thủ Đức, Hồ Chí Minh",15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,"Quận 12, Hồ Chí Minh",NaN,NaN,NaN,Nhà Đất Thổ Cư
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,"Quận 3, Hồ Chí Minh",4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,"Quận 2, Hồ Chí Minh",1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư
...,...,...,...,...,...,...,...,...
71227,"Chính chủ cần bán gấp căn hộ 4PN, 141m2, căn g...",Giá thỏa thuận,141 m²,"Thanh Trì, Hà Nội",4 Phòng ngủ,3 WC,NaN,Căn Hộ Chung Cư
71228,Chính chủ cần bán gấp căn 3 ngủ FLC Cầu Giấy g...,"7,5 tỷ",98 m²,"Cầu Giấy, Hà Nội",3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư
71229,"Bán CC 3PN 2WC tại AZ Lâm Viên Complex, giá 9,...","9,7 tỷ",129 m²,"Cầu Giấy, Hà Nội",3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư
71230,"Cần bán CHCC Hòa Bình Green Apartment, Vĩnh Ph...","6,75 tỷ",90 m²,"Ba Đình, Hà Nội",2 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư


In [130]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71232 entries, 0 to 71231
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Tên dự án    71232 non-null  object 
 1   Giá          71232 non-null  object 
 2   Diện tích    71232 non-null  object 
 3   Vị trí       71232 non-null  object 
 4   Phòng ngủ    61954 non-null  object 
 5   Nhà vệ sinh  58440 non-null  object 
 6   Ngày đăng    0 non-null      float64
 7   Loại BĐS     71232 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [131]:
# Tách quận/huyện từ cột "Vị trí" (giả sử định dạng "Quận/Huyện, Tỉnh/Thành phố")
df1['Quận/Huyện'] = df1['Vị trí'].str.split(',').str[0].str.strip()

# Tách tỉnh/thành phố từ cột "Vị trí"
df1['Tỉnh/Thành phố'] = df1['Vị trí'].str.split(',').str[1].str.strip()

# Xoá cột "Vị trí" sau khi đã tách
df1.drop(columns=['Vị trí'], inplace=True)

In [132]:
df1

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
71227,"Chính chủ cần bán gấp căn hộ 4PN, 141m2, căn g...",Giá thỏa thuận,141 m²,4 Phòng ngủ,3 WC,NaN,Căn Hộ Chung Cư,Thanh Trì,Hà Nội
71228,Chính chủ cần bán gấp căn 3 ngủ FLC Cầu Giấy g...,"7,5 tỷ",98 m²,3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Cầu Giấy,Hà Nội
71229,"Bán CC 3PN 2WC tại AZ Lâm Viên Complex, giá 9,...","9,7 tỷ",129 m²,3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Cầu Giấy,Hà Nội
71230,"Cần bán CHCC Hòa Bình Green Apartment, Vĩnh Ph...","6,75 tỷ",90 m²,2 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Ba Đình,Hà Nội


### Xử lý file csv

In [133]:
# Đọc file và gán tỉnh/thành phố
files1 = {
    'Nha_dat_BinhDuong.csv' : 'Bình Dương',
    'Nha_dat_DaNang.csv' : 'Đà Nẵng',
    'Nha_dat_DongNai.csv': 'Đồng Nai',
    'Nha_dat_HaNoi.csv' : 'Hà Nội',
    'Nha_dat_HCM.csv' : 'Hồ Chí Minh',
}

# Đọc tất cả các file CSV và gộp chúng lại
dfs1 = []

for file, tinh in files1.items():
    dfr2 = pd.read_csv(file)
    dfr2['Tỉnh/Thành phố'] = tinh
    dfs1.append(dfr2)

# Gộp tất cả thành một DataFrame
df2 = pd.concat(dfs1, ignore_index=True)
df2

,Tên dự án,Giá,Diện tích,Loại nhà,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Trang,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,Nhà Đất Thổ Cư,Huyện Dầu Tiếng,NaN,NaN,"Hôm nay, 1 giờ 20 phút trước.",1,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,Nhà Mặt Phố,Thành Phố Thủ Dầu Một,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",1,Bình Dương
...,...,...,...,...,...,...,...,...,...,...
291976,MT NGUYỄN MINH CHÂU 4.5 X 13.5M NHÀ 4 TẦNG BTC...,8.3 Tỷ,60 M²,Nhà Mặt Phố,Nguyễn Minh Châu,4 Phòng ngủ,4 WC,"Hôm nay, 29 phút trước.",3208,Hồ Chí Minh
291977,NHÀ MỚI QUẬN 10 40M² FULL NỘI THẤT 8.2 TỶ - GẦ...,8.2 Tỷ,37 M²,Nhà Trong Ngõ,Ngô Gia Tự,4 Phòng ngủ,4 WC,"Hôm nay, 8 phút trước.",3209,Hồ Chí Minh
291978,BÁN NHÀ CĂN GÓC 5 TẦNG MẶT TIỀN KINH DOANH ĐƯỜ...,29 Tỷ,88 M²,Nhà Mặt Phố,Quận 6,9 Phòng ngủ,9 WC,"Hôm nay, 22 phút trước.",3209,Hồ Chí Minh
291979,BÁN KHÁCH SẠN ĐANG KD 21 PHÒNG TẠI P2 QUẬN 6,30 Tỷ,100 M²,Nhà Mặt Phố,Quận 6,11 Phòng ngủ,11 WC,"Hôm nay, 21 phút trước.",3209,Hồ Chí Minh


In [134]:
# Đổi tên cột "Vị trí" thành "Quận/Huyện"
df2.rename(columns={'Vị trí': 'Quận/Huyện', 'Loại nhà' : 'Loại BĐS'}, inplace=True)

In [135]:
# sắp xếp lại thứ tự cột cho trùng với df1
df2 = df2[['Tên dự án', 'Giá', 'Diện tích', 'Phòng ngủ', 'Nhà vệ sinh', 'Ngày đăng', 'Loại BĐS', 'Quận/Huyện', 'Tỉnh/Thành phố']]
df2

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,NaN,NaN,"Hôm nay, 1 giờ 20 phút trước.",Nhà Đất Thổ Cư,Huyện Dầu Tiếng,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",Nhà Mặt Phố,Thành Phố Thủ Dầu Một,Bình Dương
...,...,...,...,...,...,...,...,...,...
291976,MT NGUYỄN MINH CHÂU 4.5 X 13.5M NHÀ 4 TẦNG BTC...,8.3 Tỷ,60 M²,4 Phòng ngủ,4 WC,"Hôm nay, 29 phút trước.",Nhà Mặt Phố,Nguyễn Minh Châu,Hồ Chí Minh
291977,NHÀ MỚI QUẬN 10 40M² FULL NỘI THẤT 8.2 TỶ - GẦ...,8.2 Tỷ,37 M²,4 Phòng ngủ,4 WC,"Hôm nay, 8 phút trước.",Nhà Trong Ngõ,Ngô Gia Tự,Hồ Chí Minh
291978,BÁN NHÀ CĂN GÓC 5 TẦNG MẶT TIỀN KINH DOANH ĐƯỜ...,29 Tỷ,88 M²,9 Phòng ngủ,9 WC,"Hôm nay, 22 phút trước.",Nhà Mặt Phố,Quận 6,Hồ Chí Minh
291979,BÁN KHÁCH SẠN ĐANG KD 21 PHÒNG TẠI P2 QUẬN 6,30 Tỷ,100 M²,11 Phòng ngủ,11 WC,"Hôm nay, 21 phút trước.",Nhà Mặt Phố,Quận 6,Hồ Chí Minh


### Xử lý file json

In [136]:
# Đọc file và gán tỉnh/thành phố
files2 = {
    'Nha_dat_BinhDuong.json' : 'Bình Dương',
    'Nha_dat_DaNang.json' : 'Đà Nẵng',
    'Nha_dat_DongNai.json': 'Đồng Nai',
    'Nha_dat_HaNoi.json' : 'Hà Nội',
    'Nha_dat_HCM.json' : 'Hồ Chí Minh',
    'Nha_dat_HN_total_711.json' : 'Hà Nội'
}

# Đọc tất cả các file CSV và gộp chúng lại
dfs2 = []

for file, tinh in files2.items():
    dfr3 = pd.read_json(file)
    dfr3['Tỉnh/Thành phố'] = tinh
    dfs2.append(dfr3)

# Gộp tất cả thành một DataFrame
df3 = pd.concat(dfs2, ignore_index=True)
df3

,Tên dự án,Giá,Diện tích,Loại nhà,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Trang,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,Nhà Đất Thổ Cư,Huyện Dầu Tiếng,,,"Hôm nay, 1 giờ 20 phút trước.",1,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,Nhà Mặt Phố,Thành Phố Thủ Dầu Một,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",1,Bình Dương
...,...,...,...,...,...,...,...,...,...,...
251166,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,Nhà Mặt Phố,Quận Hoàn Kiếm,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",2470,Hà Nội
251167,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,Nhà Trong Ngõ,Quận Ba Đình,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",2470,Hà Nội
251168,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,"Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,,,"15/6/2024, lúc: 14 giờ 21 phút",2471,Hà Nội
251169,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,Nhà Đất Thổ Cư,Quận Hoàng Mai,,,"15/6/2024, lúc: 14 giờ 11 phút",2471,Hà Nội


In [137]:
# Đổi tên cột "Vị trí" thành "Quận/Huyện"
df3.rename(columns={'Vị trí': 'Quận/Huyện', 'Loại nhà' : 'Loại BĐS'}, inplace=True)

In [138]:
# sắp xếp lại thứ tự cột cho trùng với df1
df3 = df3[['Tên dự án', 'Giá', 'Diện tích', 'Phòng ngủ', 'Nhà vệ sinh', 'Ngày đăng', 'Loại BĐS', 'Quận/Huyện', 'Tỉnh/Thành phố']]
df3

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,,,"Hôm nay, 1 giờ 20 phút trước.",Nhà Đất Thổ Cư,Huyện Dầu Tiếng,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",Nhà Mặt Phố,Thành Phố Thủ Dầu Một,Bình Dương
...,...,...,...,...,...,...,...,...,...
251166,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Mặt Phố,Quận Hoàn Kiếm,Hà Nội
251167,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Trong Ngõ,Quận Ba Đình,Hà Nội
251168,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút","Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,Hà Nội
251169,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội


### Hợp nhất các file

In [ ]:
# Gộp 3 DataFrame df1 df2 df3
df = pd.concat([df1, df2, df3], ignore_index=True)
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
614379,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Mặt Phố,Quận Hoàn Kiếm,Hà Nội
614380,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Trong Ngõ,Quận Ba Đình,Hà Nội
614381,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút","Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,Hà Nội
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội


## làm sạch các dòng không phải nhà ở và chuẩn hóa lại cột loại bất động sản

In [140]:
def chuan_hoa_loai_bds(loai):
    loai = str(loai).lower()
    if any(kw in loai for kw in ['nhà mặt phố', 'nhà trong ngõ', 'nhà biệt thự']):
        return 'nhà đất thổ cư'
    elif 'căn hộ' in loai:
        return 'căn hộ chung cư'
    elif 'thổ cư' in loai:
        return 'nhà đất thổ cư'
    return None  # Loại khác sẽ bị loại bỏ

# Áp dụng chuẩn hóa
df['Loại BĐS chuẩn'] = df['Loại BĐS'].apply(chuan_hoa_loai_bds)

# Chỉ giữ lại các dòng hợp lệ
df = df[df['Loại BĐS chuẩn'].isin(['căn hộ chung cư', 'nhà đất thổ cư'])]

# Gán ngược nếu muốn thay cột cũ
df['Loại BĐS'] = df['Loại BĐS chuẩn']

# Xoá cột "Loại BĐS chuẩn" sau khi đã gán ngược
df.drop(columns='Loại BĐS chuẩn', inplace=True)

# Hàm loại bỏ dấu tiếng Việt
def remove_vietnamese_accents(text):
    if pd.isna(text):
        return ''
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(text))
        if unicodedata.category(c) != 'Mn'
    )

# Tạo cột tạm đã loại dấu
df['Tên dự án không dấu'] = df['Tên dự án'].apply(remove_vietnamese_accents)

# Lọc bỏ các dòng chứa từ "toa" (không dấu, không phân biệt hoa thường)
df = df[~df['Tên dự án không dấu'].str.contains('toa', case=False, na=False)]

# Xoá cột tạm nếu không cần
df.drop(columns='Tên dự án không dấu', inplace=True)

df


C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\1708564922.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Loại BĐS'] = df['Loại BĐS chuẩn']
C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\1708564922.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns='Loại BĐS chuẩn', inplace=True)
C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\1708564922.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: 

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,nhà đất thổ cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",nhà đất thổ cư,Quận Hoàng Mai,Hà Nội
614380,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",nhà đất thổ cư,Quận Ba Đình,Hà Nội
614381,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",nhà đất thổ cư,Quận Cầu Giấy,Hà Nội
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",nhà đất thổ cư,Quận Hoàng Mai,Hà Nội


## làm sạch và chuẩn hóa cột thời gian đăng

In [141]:
# Hàm để trích xuất năm từ chuỗi ngày
def process_ngay_dang(value):
    if pd.isna(value) or '/' not in str(value):
        # Random ngày từ 1/1/2025 đến hiện tại
        start_date = datetime(2025, 1, 1)
        end_date = datetime.now()
        random_date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
        return random_date.date()
    else:
        # Tách lấy phần trước dấu phẩy và parse thành datetime
        try:
            date_part = value.split(',')[0].strip()
            parsed_date = pd.to_datetime(date_part, dayfirst=True, errors='coerce')
            return parsed_date.date() if pd.notna(parsed_date) else np.nan
        except:
            return np.nan

# Tạo cột mới đã xử lý
df['Thời gian đăng'] = df['Ngày đăng'].apply(process_ngay_dang)
df['Thời gian đăng'] = pd.to_datetime(df['Thời gian đăng'], errors='coerce')
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Quận 9,Hồ Chí Minh,2025-03-21
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Thủ Đức,Hồ Chí Minh,2025-05-31
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,nhà đất thổ cư,Quận 12,Hồ Chí Minh,2025-06-15
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh,2025-05-11
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh,2025-04-03
...,...,...,...,...,...,...,...,...,...,...
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",nhà đất thổ cư,Quận Hoàng Mai,Hà Nội,2024-06-15
614380,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",nhà đất thổ cư,Quận Ba Đình,Hà Nội,2024-06-15
614381,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",nhà đất thổ cư,Quận Cầu Giấy,Hà Nội,2024-06-15
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",nhà đất thổ cư,Quận Hoàng Mai,Hà Nội,2024-06-15


In [142]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 507090 entries, 0 to 614383
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Tên dự án       507090 non-null  object        
 1   Giá             507090 non-null  object        
 2   Diện tích       507090 non-null  object        
 3   Phòng ngủ       413140 non-null  object        
 4   Nhà vệ sinh     397961 non-null  object        
 5   Ngày đăng       441523 non-null  object        
 6   Loại BĐS        507090 non-null  object        
 7   Quận/Huyện      507090 non-null  object        
 8   Tỉnh/Thành phố  507090 non-null  object        
 9   Thời gian đăng  507090 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(9)
memory usage: 42.6+ MB


## Làm sạch và chuẩn hóa cột giá

In [ ]:
# Lọc chỉ giữ lại các dòng chứa 'tỷ' hoặc 'triệu'
df = df[df['Giá'].astype(str).str.contains('tỷ|triệu', case=False, na=False)]

# Loại các dòng có từ 2 dấu chấm trở lên
df = df[df['Giá'].astype(str).str.count(r'\.') < 2]

# Loại trùng theo 'Tên dự án'
df.drop_duplicates(subset=['Tên dự án'], inplace=True)

# Tách giá trị số
def extract_price(x):
    if isinstance(x, str):
        match = re.search(r'\d+(?:[.,]\d+)?', x)
        if match:
            return float(match.group(0).replace(',', '.'))
    return None

df['Giá (tỷ đồng)'] = df['Giá'].apply(extract_price)

# Tách đơn vị
df['dvt'] = df['Giá'].apply(
    lambda x: x.split()[1] if isinstance(x, str) and len(x.split()) > 1 else None
)

# Chuyển triệu → tỷ
df.loc[df['dvt'].str.lower() == 'triệu', 'Giá (tỷ đồng)'] /= 1000

# Lọc giá trị > 0.1 tỷ
df = df[df['Giá (tỷ đồng)'] > 0.1]

#Xử lý outlier bằng IQR
Q1 = df['Giá (tỷ đồng)'].quantile(0.25)
Q3 = df['Giá (tỷ đồng)'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df['Giá (tỷ đồng)'] >= lower_bound) & (df['Giá (tỷ đồng)'] <= upper_bound)]


In [144]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Quận 9,Hồ Chí Minh,2025-03-21,18.200,tỷ
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,nhà đất thổ cư,Thủ Đức,Hồ Chí Minh,2025-05-31,17.000,tỷ
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,nhà đất thổ cư,Quận 12,Hồ Chí Minh,2025-06-15,16.000,tỷ
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh,2025-05-11,2.450,tỷ
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh,2025-04-03,4.950,tỷ
...,...,...,...,...,...,...,...,...,...,...,...,...
587614,NHÀ MỚI ĐẠI CÁT ĐÓN TẾT - 5 TẦNG PHÂN LÔ - NỘI...,4.1 Tỷ,35 M²,3 Phòng ngủ,4 WC,"10/1/2025, lúc: 15 giờ 39 phút",nhà đất thổ cư,Quận Bắc Từ Liêm,Hà Nội,2025-01-10,4.100,Tỷ
587615,CHUNG CƯ ĐẶNG XÁ GIA LÂM. KHUÔN TIỀN VỪA PHẢI,120 Triệu,60 M²,2 Phòng ngủ,1 WC,"10/1/2025, lúc: 15 giờ 36 phút",căn hộ chung cư,Chung Cư Đặng Xá,Hà Nội,2025-01-10,0.120,Triệu
587616,KHÁCH HÀNG QUAN TÂM NHANH NHÉ. NHÀ ĐẸP Ở SƯỚNG...,186 Triệu,31 M²,3 Phòng ngủ,3 WC,"10/1/2025, lúc: 15 giờ 34 phút",nhà đất thổ cư,Bồ Đề,Hà Nội,2025-01-10,0.186,Triệu
587617,"BÁN CĂN HỘ STARLAKE TÂY HỒ 115M2 3PN 2VS 14,5 ...",14.5 Tỷ,115 M²,3 Phòng ngủ,2 WC,"10/1/2025, lúc: 15 giờ 32 phút",căn hộ chung cư,Star Lake Tây Hồ Tây,Hà Nội,2025-01-10,14.500,Tỷ


## Làm sạch cột diện tích

In [145]:
# Loại bỏ các dòng có giá trị thiếu (NaN) trong các cột quan trọng
df.dropna(subset=['Diện tích'], inplace=True)

# Trích xuất diện tích từ chuỗi, chuẩn hóa dấu phẩy sang dấu chấm và chuyển sang float
df['Diện tích(m2)'] = df['Diện tích'].str.extract(r'([\d,.]+)').replace(',', '.', regex=True).astype(float)

# Loại bỏ các dòng có diện tích <= 10 m2
df = df[df['Diện tích(m2)'] > 10]

# Tính IQR cho cột "Diện tích(m2)"
Q1 = df['Diện tích(m2)'].quantile(0.25)
Q3 = df['Diện tích(m2)'].quantile(0.75)
IQR = Q3 - Q1

# Xác định ngưỡng trên và dưới
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Lọc bỏ outlier
df = df[(df['Diện tích(m2)'] >= lower_bound) & (df['Diện tích(m2)'] <= upper_bound)]

# Tính giá trung bình trên mỗi mét vuông, đơn vị: triệu đồng/m2
df['Triệu/m2'] = round(df['Giá (tỷ đồng)'] * 1000 / df['Diện tích(m2)'], 2)


In [146]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt,Diện tích(m2),Triệu/m2
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh,2025-05-11,2.450,tỷ,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh,2025-04-03,4.950,tỷ,50.0,99.00
5,TÔI KHOA CÓ CĂN NHÀ CẦN BÁN ĐƯỜNG HOÀNG HOA TH...,"2,59 tỷ","49,7 m²",4 Phòng ngủ,5 WC,NaN,nhà đất thổ cư,Bình Thạnh,Hồ Chí Minh,2025-05-22,2.590,tỷ,49.7,52.11
6,"BÁN NHÀ RIÊNG TẠI 42/18 ĐƯỜNG SỐ 3, PHƯỜNG 8, ...","9,95 tỷ","61,4 m²",4 Phòng ngủ,6 WC,NaN,nhà đất thổ cư,Gò Vấp,Hồ Chí Minh,2025-04-18,9.950,tỷ,61.4,162.05
7,BÁN CHDV KINH DOANH TỐT TẠI TÂN PHÚ GIÁ 18TỶ80...,"18,8 tỷ",150 m²,31 Phòng ngủ,31 WC,NaN,nhà đất thổ cư,Tân Phú,Hồ Chí Minh,2025-04-10,18.800,tỷ,150.0,125.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587614,NHÀ MỚI ĐẠI CÁT ĐÓN TẾT - 5 TẦNG PHÂN LÔ - NỘI...,4.1 Tỷ,35 M²,3 Phòng ngủ,4 WC,"10/1/2025, lúc: 15 giờ 39 phút",nhà đất thổ cư,Quận Bắc Từ Liêm,Hà Nội,2025-01-10,4.100,Tỷ,35.0,117.14
587615,CHUNG CƯ ĐẶNG XÁ GIA LÂM. KHUÔN TIỀN VỪA PHẢI,120 Triệu,60 M²,2 Phòng ngủ,1 WC,"10/1/2025, lúc: 15 giờ 36 phút",căn hộ chung cư,Chung Cư Đặng Xá,Hà Nội,2025-01-10,0.120,Triệu,60.0,2.00
587616,KHÁCH HÀNG QUAN TÂM NHANH NHÉ. NHÀ ĐẸP Ở SƯỚNG...,186 Triệu,31 M²,3 Phòng ngủ,3 WC,"10/1/2025, lúc: 15 giờ 34 phút",nhà đất thổ cư,Bồ Đề,Hà Nội,2025-01-10,0.186,Triệu,31.0,6.00
587617,"BÁN CĂN HỘ STARLAKE TÂY HỒ 115M2 3PN 2VS 14,5 ...",14.5 Tỷ,115 M²,3 Phòng ngủ,2 WC,"10/1/2025, lúc: 15 giờ 32 phút",căn hộ chung cư,Star Lake Tây Hồ Tây,Hà Nội,2025-01-10,14.500,Tỷ,115.0,126.09


In [147]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
Index: 166488 entries, 3 to 587618
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Tên dự án       166488 non-null  object        
 1   Giá             166488 non-null  object        
 2   Diện tích       166488 non-null  object        
 3   Phòng ngủ       121120 non-null  object        
 4   Nhà vệ sinh     111775 non-null  object        
 5   Ngày đăng       117220 non-null  object        
 6   Loại BĐS        166488 non-null  object        
 7   Quận/Huyện      166488 non-null  object        
 8   Tỉnh/Thành phố  166488 non-null  object        
 9   Thời gian đăng  166488 non-null  datetime64[ns]
 10  Giá (tỷ đồng)   166488 non-null  float64       
 11  dvt             166488 non-null  object        
 12  Diện tích(m2)   166488 non-null  float64       
 13  Triệu/m2        166488 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(

## Làm sạch cột phòng ngủ và nhà vệ sinh

In [148]:
# Tách và chuyển đổi số phòng ngủ từ chuỗi sang kiểu float
df['Phòng ngủ'] = df['Phòng ngủ'].str.extract(r'(\d+)').astype(float)

# Tách và chuyển đổi số nhà vệ sinh từ chuỗi sang kiểu float
df['Nhà vệ sinh'] = df['Nhà vệ sinh'].str.extract(r'(\d+)').astype(float)

def fill_missing_rooms_by_area(df, bin_size=100):
    # 2. Chia nhóm diện tích dựa trên toàn bộ file
    max_area = int(df['Diện tích(m2)'].max())
    bins = list(range(0, (max_area + bin_size), bin_size))
    df['Nhóm diện tích'] = pd.cut(df['Diện tích(m2)'], bins=bins)

    # 3. Tính trung bình phòng ngủ và vệ sinh trong từng nhóm
    mean_vals = df.groupby('Nhóm diện tích')[['Phòng ngủ', 'Nhà vệ sinh']].mean().round(0)

    # 4. Hàm nội bộ để điền từng giá trị bị thiếu
    def fill_from_group(row, col):
        if pd.isna(row[col]):
            group = row['Nhóm diện tích']
            if pd.isna(group) or group not in mean_vals.index:
                return row[col]  # nếu không có nhóm thì trả lại nguyên
            return round(mean_vals.loc[group, col])
        return row[col]

    # 5. Áp dụng
    df['Phòng ngủ'] = df.apply(lambda r: fill_from_group(r, 'Phòng ngủ'), axis=1)
    df['Nhà vệ sinh'] = df.apply(lambda r: fill_from_group(r, 'Nhà vệ sinh'), axis=1)

    # 6. Xoá cột nhóm diện tích
    df.drop(columns='Nhóm diện tích', inplace=True)
    return df

# Áp dụng hàm để điền giá trị thiếu
df = fill_missing_rooms_by_area(df)

C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\973707113.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_vals = df.groupby('Nhóm diện tích')[['Phòng ngủ', 'Nhà vệ sinh']].mean().round(0)


In [149]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt,Diện tích(m2),Triệu/m2
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4.0,3.0,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh,2025-05-11,2.450,tỷ,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1.0,1.0,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh,2025-04-03,4.950,tỷ,50.0,99.00
5,TÔI KHOA CÓ CĂN NHÀ CẦN BÁN ĐƯỜNG HOÀNG HOA TH...,"2,59 tỷ","49,7 m²",4.0,5.0,NaN,nhà đất thổ cư,Bình Thạnh,Hồ Chí Minh,2025-05-22,2.590,tỷ,49.7,52.11
6,"BÁN NHÀ RIÊNG TẠI 42/18 ĐƯỜNG SỐ 3, PHƯỜNG 8, ...","9,95 tỷ","61,4 m²",4.0,6.0,NaN,nhà đất thổ cư,Gò Vấp,Hồ Chí Minh,2025-04-18,9.950,tỷ,61.4,162.05
7,BÁN CHDV KINH DOANH TỐT TẠI TÂN PHÚ GIÁ 18TỶ80...,"18,8 tỷ",150 m²,31.0,31.0,NaN,nhà đất thổ cư,Tân Phú,Hồ Chí Minh,2025-04-10,18.800,tỷ,150.0,125.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587614,NHÀ MỚI ĐẠI CÁT ĐÓN TẾT - 5 TẦNG PHÂN LÔ - NỘI...,4.1 Tỷ,35 M²,3.0,4.0,"10/1/2025, lúc: 15 giờ 39 phút",nhà đất thổ cư,Quận Bắc Từ Liêm,Hà Nội,2025-01-10,4.100,Tỷ,35.0,117.14
587615,CHUNG CƯ ĐẶNG XÁ GIA LÂM. KHUÔN TIỀN VỪA PHẢI,120 Triệu,60 M²,2.0,1.0,"10/1/2025, lúc: 15 giờ 36 phút",căn hộ chung cư,Chung Cư Đặng Xá,Hà Nội,2025-01-10,0.120,Triệu,60.0,2.00
587616,KHÁCH HÀNG QUAN TÂM NHANH NHÉ. NHÀ ĐẸP Ở SƯỚNG...,186 Triệu,31 M²,3.0,3.0,"10/1/2025, lúc: 15 giờ 34 phút",nhà đất thổ cư,Bồ Đề,Hà Nội,2025-01-10,0.186,Triệu,31.0,6.00
587617,"BÁN CĂN HỘ STARLAKE TÂY HỒ 115M2 3PN 2VS 14,5 ...",14.5 Tỷ,115 M²,3.0,2.0,"10/1/2025, lúc: 15 giờ 32 phút",căn hộ chung cư,Star Lake Tây Hồ Tây,Hà Nội,2025-01-10,14.500,Tỷ,115.0,126.09


## Chuẩn hóa lại cột Quận/Huyện

In [150]:
danh_sach_quan_huyen = {
    "Hồ Chí Minh": [
        "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8", "Quận 9", "Quận 10", "Quận 11", "Quận 12",
        "Bình Thạnh", "Phú Nhuận", "Tân Bình", "Tân Phú", "Gò Vấp", "Bình Tân", "Thủ Đức", 
        "Hóc Môn", "Củ Chi", "Bình Chánh", "Nhà Bè", "Cần Giờ"
    ],
    "Hà Nội": [
        "Ba Đình", "Hoàn Kiếm", "Tây Hồ", "Long Biên", "Cầu Giấy", "Đống Đa", "Hai Bà Trưng", "Hoàng Mai", "Thanh Xuân",
        "Sóc Sơn", "Đông Anh", "Gia Lâm", "Nam Từ Liêm", "Bắc Từ Liêm", "Thanh Trì", "Hoài Đức", "Quốc Oai", "Chương Mỹ",
        "Thường Tín", "Phú Xuyên", "Mê Linh", "Hà Đông", "Sơn Tây", "Ba Vì", "Phúc Thọ", "Đan Phượng", "Thạch Thất", "Ứng Hòa", "Mỹ Đức"
    ],
    "Bình Dương": [
        "Thành phố Thủ Dầu Một", "Thành phố Dĩ An", "Thành phố Thuận An", "Thị xã Bến Cát", "Thị xã Tân Uyên",
        "Huyện Bàu Bàng", "Huyện Bắc Tân Uyên", "Huyện Dầu Tiếng", "Huyện Phú Giáo"
    ],
    "Đồng Nai": [
        "Thành phố Biên Hòa", "Thành phố Long Khánh", "Huyện Tân Phú", "Huyện Vĩnh Cửu", "Huyện Định Quán", "Huyện Thống Nhất",
        "Huyện Trảng Bom", "Huyện Cẩm Mỹ", "Huyện Long Thành", "Huyện Nhơn Trạch", "Huyện Xuân Lộc"
    ],
    "Đà Nẵng": [
        "Quận Hải Châu", "Quận Thanh Khê", "Quận Sơn Trà", "Quận Ngũ Hành Sơn", "Quận Liên Chiểu", "Quận Cẩm Lệ",
        "Huyện Hòa Vang", "Huyện Hoàng Sa"
    ]
}

# Tạo từ điển lowercase để dễ kiểm tra
danh_sach_quan_huyen_lower = {
    tinh: [qh.lower() for qh in ds] for tinh, ds in danh_sach_quan_huyen.items()
}

# Hàm chuẩn hóa chuỗi: bỏ dấu, viết thường, xóa khoảng trắng thừa
def normalize_text(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join([c for c in text if not unicodedata.combining(c)])
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

def chuan_hoa_quan_huyen(row):
    tinh = row['Tỉnh/Thành phố']
    qh_text = str(row['Quận/Huyện'])

    if tinh not in danh_sach_quan_huyen:
        return row['Quận/Huyện']  # Không xử lý tỉnh ngoài danh sách

    norm_qh_text = normalize_text(qh_text)

    for i, qh in enumerate(danh_sach_quan_huyen[tinh]):
        norm_qh = normalize_text(qh)
        # So khớp nguyên từ bằng regex (ngăn quận 1 khớp vào quận 12)
        if re.search(r'\b{}\b'.format(re.escape(norm_qh)), norm_qh_text):
            return danh_sach_quan_huyen[tinh][i]
    return None  # Không khớp


# Tạo cột tạm chứa kết quả
df['Quận/Huyện mới'] = df.apply(chuan_hoa_quan_huyen, axis=1)

# Xoá dòng không khớp
df = df[df['Quận/Huyện mới'].notna()]

# Cập nhật lại cột gốc
df['Quận/Huyện'] = df['Quận/Huyện mới']


C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\3513001707.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Quận/Huyện'] = df['Quận/Huyện mới']


### Xóa các cột không cần thiết

In [151]:
# Xóa các cột không còn cần thiết sau xử lý
df.drop(columns=['Giá', 'dvt', 'Quận/Huyện mới'], inplace=True)


C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\2996402630.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['Giá', 'dvt', 'Quận/Huyện mới'], inplace=True)


## Đổi tên cột

In [152]:
# Đổi tên cột
df.rename(columns={'Tên dự án': 'BĐS'}, inplace=True)

C:\Users\Giang\AppData\Local\Temp\ipykernel_19728\3716704636.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'Tên dự án': 'BĐS'}, inplace=True)


In [153]:
df

,BĐS,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),Diện tích(m2),Triệu/m2
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,61 m²,4.0,3.0,NaN,nhà đất thổ cư,Quận 3,Hồ Chí Minh,2025-05-11,2.45,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,50 m²,1.0,1.0,NaN,nhà đất thổ cư,Quận 2,Hồ Chí Minh,2025-04-03,4.95,50.0,99.00
5,TÔI KHOA CÓ CĂN NHÀ CẦN BÁN ĐƯỜNG HOÀNG HOA TH...,"49,7 m²",4.0,5.0,NaN,nhà đất thổ cư,Bình Thạnh,Hồ Chí Minh,2025-05-22,2.59,49.7,52.11
6,"BÁN NHÀ RIÊNG TẠI 42/18 ĐƯỜNG SỐ 3, PHƯỜNG 8, ...","61,4 m²",4.0,6.0,NaN,nhà đất thổ cư,Gò Vấp,Hồ Chí Minh,2025-04-18,9.95,61.4,162.05
7,BÁN CHDV KINH DOANH TỐT TẠI TÂN PHÚ GIÁ 18TỶ80...,150 m²,31.0,31.0,NaN,nhà đất thổ cư,Tân Phú,Hồ Chí Minh,2025-04-10,18.80,150.0,125.33
...,...,...,...,...,...,...,...,...,...,...,...,...
587607,NHÀ MỚI Ở NGAY- DƯƠNG NỘI- FULL NỘI THẤT- 5.X TỶ,42 M²,4.0,3.0,"10/1/2025, lúc: 16 giờ 32 phút",nhà đất thổ cư,Hà Đông,Hà Nội,2025-01-10,5.90,42.0,140.48
587608,"BÁN NHÀ RIÊNG CẦU GIẤY, NGÕ THÔNG, MẶT NGÕ 35M...",36 M²,3.0,3.0,"10/1/2025, lúc: 16 giờ 19 phút",nhà đất thổ cư,Cầu Giấy,Hà Nội,2025-01-10,8.70,36.0,241.67
587612,"BÁN ĐẤT HOÀNG MAI 94M2 11.9TỶ, MẶT TIỀN 4.8M, ...",94 M²,3.0,3.0,"10/1/2025, lúc: 15 giờ 47 phút",nhà đất thổ cư,Hoàng Mai,Hà Nội,2025-01-10,11.90,94.0,126.60
587614,NHÀ MỚI ĐẠI CÁT ĐÓN TẾT - 5 TẦNG PHÂN LÔ - NỘI...,35 M²,3.0,4.0,"10/1/2025, lúc: 15 giờ 39 phút",nhà đất thổ cư,Bắc Từ Liêm,Hà Nội,2025-01-10,4.10,35.0,117.14


### Sắp xếp lại vị trí các cột cho phù hợp

In [154]:
# Sắp xếp lại thứ tự cột để phù hợp với báo cáo/hiển thị
df = df[['BĐS',
 'Loại BĐS',
 'Tỉnh/Thành phố',
 'Quận/Huyện',
 'Diện tích(m2)',
 'Phòng ngủ',
 'Nhà vệ sinh',
 'Giá (tỷ đồng)',
 'Triệu/m2',
 'Thời gian đăng',]]


In [155]:
df

,BĐS,Loại BĐS,Tỉnh/Thành phố,Quận/Huyện,Diện tích(m2),Phòng ngủ,Nhà vệ sinh,Giá (tỷ đồng),Triệu/m2,Thời gian đăng
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,nhà đất thổ cư,Hồ Chí Minh,Quận 3,61.0,4.0,3.0,2.45,40.16,2025-05-11
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,nhà đất thổ cư,Hồ Chí Minh,Quận 2,50.0,1.0,1.0,4.95,99.00,2025-04-03
5,TÔI KHOA CÓ CĂN NHÀ CẦN BÁN ĐƯỜNG HOÀNG HOA TH...,nhà đất thổ cư,Hồ Chí Minh,Bình Thạnh,49.7,4.0,5.0,2.59,52.11,2025-05-22
6,"BÁN NHÀ RIÊNG TẠI 42/18 ĐƯỜNG SỐ 3, PHƯỜNG 8, ...",nhà đất thổ cư,Hồ Chí Minh,Gò Vấp,61.4,4.0,6.0,9.95,162.05,2025-04-18
7,BÁN CHDV KINH DOANH TỐT TẠI TÂN PHÚ GIÁ 18TỶ80...,nhà đất thổ cư,Hồ Chí Minh,Tân Phú,150.0,31.0,31.0,18.80,125.33,2025-04-10
...,...,...,...,...,...,...,...,...,...,...
587607,NHÀ MỚI Ở NGAY- DƯƠNG NỘI- FULL NỘI THẤT- 5.X TỶ,nhà đất thổ cư,Hà Nội,Hà Đông,42.0,4.0,3.0,5.90,140.48,2025-01-10
587608,"BÁN NHÀ RIÊNG CẦU GIẤY, NGÕ THÔNG, MẶT NGÕ 35M...",nhà đất thổ cư,Hà Nội,Cầu Giấy,36.0,3.0,3.0,8.70,241.67,2025-01-10
587612,"BÁN ĐẤT HOÀNG MAI 94M2 11.9TỶ, MẶT TIỀN 4.8M, ...",nhà đất thổ cư,Hà Nội,Hoàng Mai,94.0,3.0,3.0,11.90,126.60,2025-01-10
587614,NHÀ MỚI ĐẠI CÁT ĐÓN TẾT - 5 TẦNG PHÂN LÔ - NỘI...,nhà đất thổ cư,Hà Nội,Bắc Từ Liêm,35.0,3.0,4.0,4.10,117.14,2025-01-10


In [156]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 116135 entries, 3 to 587617
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   BĐS             116135 non-null  object        
 1   Loại BĐS        116135 non-null  object        
 2   Tỉnh/Thành phố  116135 non-null  object        
 3   Quận/Huyện      116135 non-null  object        
 4   Diện tích(m2)   116135 non-null  float64       
 5   Phòng ngủ       116135 non-null  float64       
 6   Nhà vệ sinh     116135 non-null  float64       
 7   Giá (tỷ đồng)   116135 non-null  float64       
 8   Triệu/m2        116135 non-null  float64       
 9   Thời gian đăng  116135 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 9.7+ MB


### Lưu file

In [157]:
df.to_csv('batdongsan_clean.csv', index=False, float_format='%.1f', encoding='utf-8')